In [1]:
from pathlib import Path

ROOT = Path(".").resolve().parents[1]
import sys

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
from rich import print as rprint

In [3]:
from dotenv import load_dotenv

load_dotenv("../.env")

True

In [4]:
from src.application.contracts import PipelineRequest
from src.application import ViRAGEPipeline
from src.application.settings import ViRAGESettings

tmp_path = Path("../demo_data/tmp_folder").resolve()
data_path = Path("../demo_data/Iris.csv").resolve()
settings = ViRAGESettings(artifact_root=tmp_path / "artifacts")
pipeline = ViRAGEPipeline(settings)
query = "Show the sales trend over time"
request = PipelineRequest(query=query, data_path=data_path.as_posix())

In [5]:
from langchain_ollama import ChatOllama

LLM_MODEL = "gemma3:1b"  # или "llama3.2:1b"
# LLM_MODEL = "llama3.2:1b"       # или "llama3.2:1b"
llm = ChatOllama(
    model=LLM_MODEL,
    temperature=0,
)

In [6]:
from src.infrastructure import RuntimeContext

runtime = RuntimeContext(settings=settings, llm=llm)

In [7]:
from src.services import QueryUnderstandingService

qu = QueryUnderstandingService().invoke(runtime=runtime, user_context=request.user_context, query=request.query)

In [8]:
rprint("query:", request.query)
rprint(qu)

query: Show the sales trend over time

QueryUnderstandingResult(
    intent='Show the sales trend over time',
    requested_operations=['trend analysis'],
    candidate_charts=['line'],
    constraints=[],
    case_type=<ChartCaseType.CANONICAL: 'canonical'>,
    confidence=0.8
)

In [9]:
from src.services import CanonicalPlanningService

cp = CanonicalPlanningService().invoke(runtime=runtime, query_understanding=qu)

In [10]:
rprint(cp)

PlanningResult(
    mode=<ChartCaseType.CANONICAL: 'canonical'>,
    steps=[
        PlanningStep(
            name='profile_the_dataset_and_confirm_field',
            description='Profile the dataset and confirm field types relevant to the request.'
        ),
        PlanningStep(
            name='prepare_a_cleaned_analysis_ready_version',
            description='Prepare a cleaned analysis-ready version of the data.'
        ),
        PlanningStep(
            name='retrieve_concise_charting_guidance_for_the',
            description='Retrieve concise charting guidance for the selected chart family.'
        ),
        PlanningStep(
            name='build_the_primary_requested_chart_using',
            description='Build the primary requested chart using the leading chart family: line.'
        ),
        PlanningStep(
            name='execute_plotting_code_and_collect_numeric',
            description='Execute plotting code and collect numeric summaries from the run.'
        ),
        PlanningStep(
            name='read_chart_structure,_extract_facts_and',
            description='Read chart structure, extract facts and verify that final statements are evidence-backed.'
        )
    ],
    success_criteria=[
        'At least one valid canonical chart is produced, preferably among: line.',
        'Generated charts are readable and consistent with the request.',
        'Final statements reference execution metrics or chart evidence.'
    ]
)

In [11]:
from src.services import DataProfilerService

data_profile = DataProfilerService().invoke(runtime=runtime, data_path=data_path)

In [12]:
rprint(data_profile)

DataProfile(
    row_count=150,
    col_count=6,
    columns=[
        DataColumnProfile(name='Id', dtype='numeric', missing_ratio=0.0, unique_count=150),
        DataColumnProfile(name='SepalLengthCm', dtype='numeric', missing_ratio=0.0, unique_count=35),
        DataColumnProfile(name='SepalWidthCm', dtype='numeric', missing_ratio=0.0, unique_count=23),
        DataColumnProfile(name='PetalLengthCm', dtype='numeric', missing_ratio=0.0, unique_count=43),
        DataColumnProfile(name='PetalWidthCm', dtype='numeric', missing_ratio=0.0, unique_count=22),
        DataColumnProfile(name='Species', dtype='categorical', missing_ratio=0.0, unique_count=3)
    ],
    likely_numeric_columns=['Id', 'SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm'],
    likely_categorical_columns=['Species'],
    likely_time_columns=[],
    quality_notes=["Column 'Id' looks like an identifier."]
)

In [13]:
from src.services import DataPreparationService

data_prep = DataPreparationService().invoke(runtime=runtime, data_path=data_path, data_profile=data_profile, run_id="1")

In [14]:
rprint(data_prep)

DataPreparationResult(
    output_path='D:/programming/projects/mas_rag/demo_data/tmp_folder/artifacts/1/cleaned_data.csv',
    operations=[],
    row_count=150,
    col_count=6
)

# VisRAG

In [17]:
from src.services import VisRAGService

recommendations = VisRAGService().invoke(runtime=runtime, data_profile=data_profile, query_understanding=qu, planning=cp)

In [18]:
rprint(recommendations)

VisRAGResult(
    recommendations=[
        VisRAGRecommendation(
            chart_family='line',
            rationale='Line retained as a trend fallback.',
            priority=1,
            score=0.55,
            support_examples=[],
            instruction_highlights=['Use index on the x-axis.', 'Use Id on the y-axis.']
        )
    ],
    visualization_plan=VisualizationPlan(
        chart_family='line',
        visual_task='trend analysis',
        goal='Show the sales trend over time',
        title='Show the sales trend over time',
        subtitle='Mode: canonical; preferred chart family: line.',
        description='Build a line chart for trend analysis. Preserve explicit encodings so the plan can be 
converted to code or Vega-Lite later.',
        field_bindings=[
            VisualizationFieldBinding(
                channel='x',
                field_name='Species',
                field_role='nominal',
                title='Species',
                aggregate=None,
                time_unit=None,
                sort=None,
                required=True
            ),
            VisualizationFieldBinding(
                channel='y',
                field_name='Id',
                field_role='quantitative',
                title='Id',
                aggregate=None,
                time_unit=None,
                sort=None,
                required=True
            )
        ],
        transforms=[
            VisualizationTransform(
                kind='aggregate',
                field_name='Id',
                expression=None,
                aggregate='mean',
                group_by=['Species'],
                order_by=None,
                descending=False,
                description='Aggregate the numeric measure by the temporal or ordered x axis before plotting.'
            )
        ],
        axes=[
            VisualizationAxisInstruction(
                channel='x',
                field_name='Species',
                title='Species',
                scale_type='categorical',
                format_hint=None,
                rotate_labels=True
            ),
            VisualizationAxisInstruction(
                channel='y',
                field_name='Id',
                title='Id',
                scale_type='linear',
                format_hint=None,
                rotate_labels=False
            )
        ],
        filters=[],
        build_instructions=[
            "Bind the x-axis to 'Species' and treat it as ordered categorical.",
            "Bind the y-axis to 'Id' as the primary quantitative measure.",
            "Aggregate 'Id' by 'Species' with mean when duplicate x values appear.",
            'Planning step: Profile the dataset and confirm field types relevant to the request.',
            'Planning step: Prepare a cleaned analysis-ready version of the data.',
            'Planning step: Retrieve concise charting guidance for the selected chart family.',
            'Success criterion: At least one valid canonical chart is produced, preferably among: line.',
            'Success criterion: Generated charts are readable and consistent with the request.'
        ],
        mark_hints=['Use a line mark with point markers.', 'Preserve chronological order on the x-axis.'],
        renderer_hints=['Rotate x-axis labels if dates are dense.', 'Tighten layout after formatting dates.'],
        evidence_example_ids=[],
        confidence=0.275,
        vega_lite_ready=True
    ),
    rules=[
        'Prefer specifications that can map cleanly to Vega-Lite encodings and transforms.',
        'Keep titles, axes and marks explicit so code generation stays deterministic.',
        'Prefer retrieved corpus examples when they agree with the detected data shape.'
    ],
    caveats=[
        'No local corpus matches were retrieved; VisRAG used data-aware fallback planning.',
        'LLM synthesis failed; VisRAG kept the deterministic visualization plan.'
    ],
    implementation_no

In [19]:
from pathlib import Path

from src.application.settings import ViRAGESettings
from src.infrastructure.runtime import RuntimeContext
from src.services.visrag import VisRAGService

runtime = RuntimeContext(
    # settings=ViRAGESettings(
    #     artifact_root=Path("./artifacts").resolve(),
    #     visrag_corpus_root=Path("../data/visrag_corpora").resolve(),
    #     visrag_top_k_examples=5,
    #     visrag_top_k_recommendations=3,
    #     visrag_min_example_score=0.05,
    #     llm=llm
    # )
    settings=ViRAGESettings(
        artifact_root=tmp_path / "artifacts",
        visrag_corpus_root=Path("../data/visrag_corpora").resolve(),
        visrag_rag_enabled=True,
        visrag_retriever_backend="local_tfidf",  # для ноутбука без Ollama
        visrag_retriever_top_k=3,
        visrag_retriever_fetch_k=10,
        visrag_enable_llm_refinement=False,
        llm=llm
    )
)

result = VisRAGService().invoke(query_understanding=qu, data_profile=data_profile, runtime=runtime, planning=cp)

In [22]:
rprint(result.model_dump_json(indent=2, ensure_ascii=False))

{
  "recommendations": [
    {
      "chart_family": "scatter",
      "rationale": "Two numeric fields detected; scatter suits relationship analysis. 2 retrieved corpus example(s)
support this chart family.",
      "priority": 1,
      "score": 1.007,
      "support_examples": [
        "plot2code-18b0d9536611",
        "plot2code-e0fc58fa6ae5"
      ],
      "instruction_highlights": [
        "Use Id vs SepalLengthCm."
      ]
    },
    {
      "chart_family": "line",
      "rationale": "Line retained as a trend fallback. 3 retrieved corpus example(s) support this chart family.",
      "priority": 2,
      "score": 0.7763,
      "support_examples": [
        "plot2code-3b2021d3ad70",
        "plot2code-7f3361de2144",
        "plot2code-217f692a59c8"
      ],
      "instruction_highlights": [
        "Use index on the x-axis.",
        "Use Id on the y-axis."
      ]
    }
  ],
  "visualization_plan": {
    "chart_family": "scatter",
    "visual_task": "trend analysis",
    "goal": "Show the sales trend over time",
    "title": "Show the sales trend over time",
    "subtitle": "Mode: canonical; preferred chart family: scatter.",
    "description": "Build a scatter chart for trend analysis. Preserve explicit encodings so the plan can be 
converted to code or Vega-Lite later.",
    "field_bindings": [
      {
        "channel": "x",
        "field_name": "Id",
        "field_role": "quantitative",
        "title": "Id",
        "aggregate": null,
        "time_unit": null,
        "sort": null,
        "required": true
      },
      {
        "channel": "y",
        "field_name": "SepalLengthCm",
        "field_role": "quantitative",
        "title": "SepalLengthCm",
        "aggregate": null,
        "time_unit": null,
        "sort": null,
        "required": true
      },
      {
        "channel": "color",
        "field_name": "Species",
        "field_role": "nominal",
        "title": "Species",
        "aggregate": null,
        "time_unit": null,
        "sort": null,
        "required": true
      }
    ],
    "transforms": [],
    "axes": [
      {
        "channel": "x",
        "field_name": "Id",
        "title": "Id",
        "scale_type": "linear",
        "format_hint": null,
        "rotate_labels": false
      },
      {
        "channel": "y",
        "field_name": "SepalLengthCm",
        "title": "SepalLengthCm",
        "scale_type": "linear",
        "format_hint": null,
        "rotate_labels": false
      }
    ],
    "filters": [],
    "build_instructions": [
      "Use 'Id' on the x-axis and 'SepalLengthCm' on the y-axis.",
      "Keep raw points without aggregation unless later requirements demand summarization.",
      "Use 'Species' as the grouping or color channel when plotting points.",
      "Planning step: Profile the dataset and confirm field types relevant to the request.",
      "Planning step: Prepare a cleaned analysis-ready version of the data.",
      "Planning step: Retrieve concise charting guidance for the selected chart family.",
      "Success criterion: At least one valid canonical chart is produced, preferably among: line.",
      "Success criterion: Generated charts are readable and consistent with the request."
    ],
    "mark_hints": [
      "Use a scatter mark.",
      "Consider semi-transparent points if the plot is dense."
    ],
    "renderer_hints": [],
    "evidence_example_ids": [
      "plot2code-18b0d9536611",
      "plot2code-e0fc58fa6ae5"
    ],
    "confidence": 0.503,
    "vega_lite_ready": true
  },
  "rules": [
    "Prefer specifications that can map cleanly to Vega-Lite encodings and transforms.",
    "Keep titles, axes and marks explicit so code generation stays deterministic.",
    "Prefer retrieved corpus examples when they agree with the detected data shape."
  ],
  "caveats": [
    "Semantic retrieval is running on the local TF-IDF fallback backend; enable Ollama embeddings for denser 
retrieval."
  ],
  "implementation_notes": [
    "Use 'Id' on the x-